In [ ]:
import sys

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, "..")
from config.semillas import SEMILLA_PRIMARIA

DATA_PATH = "../datasets/processed/competencia_01.csv"

con = duckdb.connect()
con.execute(f"""
    create or replace view labeled as
    select * exclude (numero_de_cliente, foto_mes)
    from read_csv_auto('{DATA_PATH}')
    where clase_ternaria is not null
""")

In [ ]:
ganancia_acierto = 1_072_500
costo_estimulo = 27_500

In [ ]:
data = pd.read_csv(DATA_PATH)

In [ ]:
mes_train = 202103
mes_test = 202105

X = data[data['foto_mes'] == mes_train]
y = X['clase_ternaria']
X = X.drop(columns=['clase_ternaria'])


In [ ]:
# import requests

# url = "https://www.random.org/integers/"
# params = {
#     "num": 100,
#     "min": 1,
#     "max": 1_000_000_000,
#     "col": 1,
#     "base": 10,
#     "format": "plain",
#     "rnd": "new"
# }

# try:
#     response = requests.get(url, params=params)
#     response.raise_for_status()
#     random_numbers_str = response.text.strip().split()
#     semillas = [int(r) for r in random_numbers_str]
# except requests.exceptions.RequestException as e:
#     print(f"Ups: {e}")
# print(semillas)

#copie y pegelas en el algun lugar, no se van a repetir estos valores en su finita existencia

semillas = [590772946, 673765742, 293750017, 943403192, 979156448, 584538485, 452449729, 558697969, 
            57360287, 130446541, 605182907, 950224428, 218289405, 939948071, 100863175, 414478288, 
            160303261, 273512111, 865860863, 213714567, 808915688, 722029742, 925723517, 945671630, 
            664520372, 54401700, 342474660, 846731114, 422760380, 578609822, 767778681, 83891579,
            527049832, 648818020, 55070016, 555906589, 258579277, 746504435, 633736736, 177637542, 
            188273233, 321487413, 786466940, 648593534, 692127408, 894074607, 198169439, 586783997,
            999893340, 749684372, 42088082, 113747440, 372302911, 134610206, 888433608, 550860016,
            15725572, 326369614, 317756119, 919179051, 627565314, 43229677, 374383345, 783812080, 
            906904853, 392284205, 162535622, 769809838, 712115623, 411443258, 634859221, 196005885,
            444086051, 797383292, 840184399, 790833327, 34990967, 888278840, 717241854, 564003472,
            526352615, 647777869, 928913153, 804729752, 432939581, 351356858, 308382688, 193177234,
            686379225, 787197794, 123401333, 501882704, 787535958, 799164038, 491576899, 285782967, 
            638402799, 695124778, 909121796, 893364907]



In [ ]:
def ganancia(model, X, y, prop=1, threshold=0.025):

  class_index = np.where(model.classes_ == "BAJA+2")[0][0]
  y_hat = model.predict_proba(X)

  @np.vectorize
  def ganancia_row(predicted, actual, threshold=0.025):
    return  (predicted >= threshold) * (ganancia_acierto if actual == "BAJA+2" else -costo_estimulo)

  return ganancia_row(y_hat[:,class_index], y).sum() / prop


In [ ]:
tree_params = {
    'criterion': 'gini',
    'max_depth': 5,
    'min_samples_split': 80
}

#lista de ganancias
gan_test = []
gan_train = [] # esta no se mira NUNCA, esta para sumar dibujitos

sss = StratifiedShuffleSplit(n_splits=10, test_size=0.3, random_state=semillas[0])

for train_index, test_index in sss.split(X, y):

  model = DecisionTreeClassifier(**tree_params)
  model.fit(X.iloc[train_index], y.iloc[train_index])

  gan_test.append(ganancia(model, X.iloc[test_index], y.iloc[test_index], prop=0.3))
  gan_train.append(ganancia(model, X.iloc[train_index], y.iloc[train_index], prop=0.7))


In [ ]:
df_gains = pd.DataFrame({
    'Ganancia': gan_test + gan_train,
    'Conjunto': ['Test'] * len(gan_test) + ['Train'] * len(gan_train)
})

g = sns.displot(
    df_gains, x='Ganancia', hue='Conjunto',
    kind='hist', kde=True, palette='viridis',
    aspect=2, height=4
)
g.set_axis_labels('Ganancia', 'Frecuencia')
g.figure.suptitle('Distribución de Ganancias por Conjunto')
g.figure.subplots_adjust(top=0.9)
plt.show()

In [ ]:
tree_params = {
    'criterion': 'gini',
    'max_depth': 5,
    'min_samples_split': 40
}

#lista de ganancias
gan_test = []
gan_train = [] # esta no se mira NUNCA, esta para sumar dibujitos

sss = StratifiedShuffleSplit(n_splits=10, test_size=0.3, random_state=semillas[0])

for train_index, test_index in sss.split(X, y):

  model = DecisionTreeClassifier(**tree_params)
  model.fit(X.iloc[train_index], y.iloc[train_index])

  gan_test.append(ganancia(model, X.iloc[test_index], y.iloc[test_index], prop=0.3))
  gan_train.append(ganancia(model, X.iloc[train_index], y.iloc[train_index], prop=0.7))

In [ ]:
df_gains = pd.DataFrame({
    'Ganancia': gan_test + gan_train,
    'Conjunto': ['Test'] * len(gan_test) + ['Train'] * len(gan_train)
})

g = sns.displot(
    df_gains, x='Ganancia', hue='Conjunto',
    kind='hist', kde=True, palette='viridis',
    aspect=2, height=4
)
g.set_axis_labels('Ganancia', 'Frecuencia')
g.figure.suptitle('Distribución de Ganancias por Conjunto')
g.figure.subplots_adjust(top=0.9)
plt.show()

In [ ]:
# Entrenamos y testeamos con marzo

mes_train = 202103
mes_test = 202103

X = data[data['foto_mes'] == mes_train]
y = X['clase_ternaria']
X = X.drop(columns=['clase_ternaria'])

tree_params = {
    'criterion': 'gini',
    'max_depth': 5,
    'min_samples_split': 80
}

#lista de ganancias
gan_test = []
gan_train = [] # esta no se mira NUNCA, esta para sumar dibujitos

sss = StratifiedShuffleSplit(n_splits=10, test_size=0.3, random_state=semillas[0])

for train_index, test_index in sss.split(X, y):

  model = DecisionTreeClassifier(**tree_params)
  model.fit(X.iloc[train_index], y.iloc[train_index])

  gan_test.append(ganancia(model, X.iloc[test_index], y.iloc[test_index], prop=0.3))
  gan_train.append(ganancia(model, X.iloc[train_index], y.iloc[train_index], prop=0.7))

df_gains = pd.DataFrame({
    'Ganancia': gan_test + gan_train,
    'Conjunto': ['Test'] * len(gan_test) + ['Train'] * len(gan_train)
})

g = sns.displot(
    df_gains, x='Ganancia', hue='Conjunto',
    kind='hist', kde=True, palette='viridis',
    aspect=2, height=4
)
g.set_axis_labels('Ganancia', 'Frecuencia')
g.figure.suptitle('Distribución de Ganancias por Conjunto')
g.figure.subplots_adjust(top=0.9)
plt.show()